# PDF → Excel 변환기 (공용차량 운행일지)
셀을 위에서부터 순서대로 실행하세요. (`Shift + Enter`)

## 1단계: 필요한 패키지 설치 (최초 1회만)

In [ ]:
!pip install pdfplumber openpyxl pandas tabula-py

## 2단계: PDF 파일 경로 설정
아래 경로를 본인의 PDF 파일/폴더 경로로 수정하세요.

In [ ]:
#===================================================
# 여기만 수정하세요!
#===================================================

# PDF 파일 또는 폴더 경로
PDF_PATH = r"C:\Users\USER\Documents\txt converter"

# 추출 엔진 선택: "pdfplumber" 또는 "tabula"
# pdfplumber가 기본값, 결과가 이상하면 tabula로 바꿔보세요
ENGINE = "pdfplumber"

#===================================================

## 3단계: 변환 실행
아래 셀을 실행하면 자동으로 변환됩니다.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter


def extract_tables_pdfplumber(pdf_path):
    import pdfplumber
    tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, 1):
            page_tables = page.extract_tables()
            if not page_tables:
                text = page.extract_text()
                if text and text.strip():
                    lines = [line.split() for line in text.strip().split("\n") if line.strip()]
                    if lines:
                        df = pd.DataFrame(lines)
                        tables.append(df)
                continue
            for table in page_tables:
                if not table:
                    continue
                cleaned = [row for row in table if any(cell and str(cell).strip() for cell in row)]
                if cleaned:
                    tables.append(pd.DataFrame(cleaned))
    return tables


def extract_tables_tabula(pdf_path):
    import tabula
    try:
        tables = tabula.read_pdf(pdf_path, pages="all", multiple_tables=True, lattice=True)
    except Exception:
        tables = []
    if not tables:
        try:
            tables = tabula.read_pdf(pdf_path, pages="all", multiple_tables=True, stream=True)
        except Exception:
            tables = []
    return [t for t in tables if not t.empty]


def promote_header(df):
    if df.empty:
        return df
    first_row = df.iloc[0]
    header_like = sum(
        1 for val in first_row
        if val is not None and str(val).strip() and not str(val).strip().replace(".", "").isdigit()
    )
    if header_like >= len(first_row) * 0.5:
        headers = [str(val).strip() if val is not None else f"Column_{i}" for i, val in enumerate(first_row)]
        df = df.iloc[1:].reset_index(drop=True)
        df.columns = headers
    return df


def merge_tables(tables):
    if not tables:
        return pd.DataFrame()
    if len(tables) == 1:
        return promote_header(tables[0])
    groups = {}
    for t in tables:
        ncols = len(t.columns)
        groups.setdefault(ncols, []).append(t)
    largest_group = max(groups.values(), key=lambda g: sum(len(t) for t in g))
    normalized = []
    for t in largest_group:
        t = t.copy()
        t.columns = range(len(t.columns))
        normalized.append(t)
    merged = pd.concat(normalized, ignore_index=True)
    return promote_header(merged)


def style_excel(wb_path):
    wb = load_workbook(wb_path)
    ws = wb.active
    header_font = Font(name="맑은 고딕", bold=True, size=11, color="FFFFFF")
    header_fill = PatternFill(start_color="2F5496", end_color="2F5496", fill_type="solid")
    cell_font = Font(name="맑은 고딕", size=10)
    thin_border = Border(
        left=Side(style="thin"), right=Side(style="thin"),
        top=Side(style="thin"), bottom=Side(style="thin"),
    )
    center_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    left_align = Alignment(horizontal="left", vertical="center", wrap_text=True)
    for row_idx, row in enumerate(ws.iter_rows(min_row=1, max_row=ws.max_row, max_col=ws.max_column), 1):
        for cell in row:
            cell.border = thin_border
            if row_idx == 1:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = center_align
            else:
                cell.font = cell_font
                cell.alignment = left_align
    for col_idx in range(1, ws.max_column + 1):
        max_length = 0
        col_letter = get_column_letter(col_idx)
        for cell in ws[col_letter]:
            if cell.value:
                val_str = str(cell.value)
                length = sum(2 if ord(c) > 127 else 1 for c in val_str)
                max_length = max(max_length, length)
        ws.column_dimensions[col_letter].width = min(max(max_length + 4, 8), 50)
    ws.freeze_panes = "A2"
    wb.save(wb_path)


def convert_pdf_to_excel(pdf_path, output_path=None, engine="pdfplumber"):
    pdf_path = str(Path(pdf_path).resolve())
    if output_path is None:
        output_path = str(Path(pdf_path).with_suffix(".xlsx"))
    if engine == "pdfplumber":
        tables = extract_tables_pdfplumber(pdf_path)
    else:
        tables = extract_tables_tabula(pdf_path)
    if not tables:
        print(f"  ⚠ 테이블을 찾지 못했습니다: {Path(pdf_path).name}")
        return None
    df = merge_tables(tables)
    df.to_excel(output_path, index=False, sheet_name="운행일지")
    style_excel(output_path)
    return output_path


print("변환 함수 준비 완료!")

In [ ]:
# 변환 실행
input_path = Path(PDF_PATH)

if input_path.is_dir():
    pdf_files = sorted(input_path.glob("*.pdf")) + sorted(input_path.glob("*.PDF"))
    print(f"폴더에서 PDF {len(pdf_files)}개 발견\n")
    results = []
    for i, pdf_file in enumerate(pdf_files, 1):
        print(f"[{i}/{len(pdf_files)}] {pdf_file.name} 변환 중...")
        result = convert_pdf_to_excel(str(pdf_file), engine=ENGINE)
        if result:
            results.append(result)
            print(f"  → {Path(result).name} 완료!")
    print(f"\n===== 총 {len(results)}개 파일 변환 완료! =====")
elif input_path.is_file():
    print(f"{input_path.name} 변환 중...")
    result = convert_pdf_to_excel(str(input_path), engine=ENGINE)
    if result:
        print(f"\n===== 완료! → {result} =====")
else:
    print(f"경로를 찾을 수 없습니다: {PDF_PATH}")
    print("2단계에서 경로를 다시 확인해주세요.")

## 4단계: 결과 미리보기 (선택사항)
변환된 첫 번째 엑셀 파일의 내용을 확인합니다.

In [ ]:
# 변환된 엑셀 파일 미리보기
if input_path.is_dir():
    xlsx_files = sorted(input_path.glob("*.xlsx"))
    if xlsx_files:
        print(f"미리보기: {xlsx_files[0].name}\n")
        df = pd.read_excel(xlsx_files[0])
        display(df)
    else:
        print("변환된 xlsx 파일이 없습니다.")
elif input_path.is_file():
    xlsx_path = input_path.with_suffix(".xlsx")
    if xlsx_path.exists():
        print(f"미리보기: {xlsx_path.name}\n")
        df = pd.read_excel(xlsx_path)
        display(df)